# Sharp Corner Microchannel (3D, XNSE-Solver with IBMelements)

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.XNSERO_Solver;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("SharpCornerMicroChannel3D_RO", myBatch);

Project name is set to 'SharpCornerMicroChannel3D_RO'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\SharpCornerMicroChannel3D_RO'.


## Problem settings

In [ ]:
double density = 1000.0;
double viscosity = 1.0e-3;
double averBulkVelocity_atContraction = 1.00; 
double averBulkVelocity_atExpansion = 0.25; 

In [ ]:
double SizeScale = 1.0e-6;
double Ls = 80.0 * SizeScale;
double Le = 200.0 * SizeScale;
double Wi = 42.5 * SizeScale;
double We = 200.0 * SizeScale;
double H = 50.0 * SizeScale;

double Linlet = 400.0 * SizeScale;

int nSections = 2;
double Lsection = Ls + Le;
double Lsystem = nSections * Lsection;

double Loutlet = 200.0 * SizeScale;

double Lend = Lsystem + Loutlet;

In [ ]:
double Dh = 2 * Wi * H / (Wi + H);  // hydraulic diameter

double Re = density * Dh * averBulkVelocity_atContraction / viscosity;
Re

45.94594594594594

In [ ]:
double Re_e = density * We * averBulkVelocity_atExpansion / viscosity;
Re

45.94594594594594

In [ ]:
double R = 100.0 * SizeScale;
double De = Math.Sqrt(H / (2.0 * R)) * Re;
De

22.97297297297297

## Grid Creation

In [ ]:
int FlowResolution = 1;

In [ ]:
int HeightResolution = 2 * FlowResolution;
double[] zNodes = GenericBlas.Linspace(-H/2.0, H/2.0, HeightResolution + 1);

In [ ]:
int ContractionCrossWiseResolution = 2 * FlowResolution;
int OuterExpansionCrossWiseResolution = 2 * FlowResolution;

In [ ]:
int InletStreamWiseResolution = 12 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(-Linlet, 0.0, InletStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid3D.Cartesian3DGrid(xNodes1, yNodes1, zNodes);

In [ ]:
double[] xNodes2 = xNodes1;
double[] yNodes2 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd2 = Grid3D.Cartesian3DGrid(xNodes2, yNodes2, zNodes);

In [ ]:
double[] xNodes3 = xNodes1;
double[] yNodes3 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid3D.Cartesian3DGrid(xNodes3, yNodes3, zNodes);

In [ ]:
var gridInlet = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3});

### repetitive sections

In [ ]:
int SectionStreamWiseResolution = 8 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(0.0, nSections * Lsection, nSections * SectionStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid3D.Cartesian3DGrid(xNodes1, yNodes1, zNodes);

In [ ]:
double[] xNodes2 = xNodes1;
double[] yNodes2 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd2 = Grid3D.Cartesian3DGrid(xNodes2, yNodes2, zNodes);

In [ ]:
double[] xNodes3 = xNodes1;
double[] yNodes3 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid3D.Cartesian3DGrid(xNodes3, yNodes3, zNodes);

In [ ]:
var gridSections = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3});
//var gridSectionSealed = GridCommons.Seal(gridSection, 4);

In [ ]:
// wedges (as IBMelements)
List<Particle> particles = new List<Particle>();
MotionFixed noMotion = new();

for (int n = 0; n < nSections; n++) {
    // lower wegde
    MultidimensionalArray pointsL = MultidimensionalArray.Create(3, 2);
    pointsL.SetSubVector(new double[] { n*Lsection, -We/2.0 }, new int[] { 0, -1 });
    pointsL.SetSubVector(new double[] { n*Lsection, -Wi/2.0 }, new int[] { 1, -1 });
    pointsL.SetSubVector(new double[] { n*Lsection + Ls, -We/2.0 }, new int[] { 2, -1 });

    particles.Add(new ParticleTriangle(noMotion, pointsL));

    // upper wegde
    MultidimensionalArray pointsU = MultidimensionalArray.Create(3, 2);
    pointsU.SetSubVector(new double[] { n*Lsection + Ls, We/2.0 }, new int[] { 0, -1 });
    pointsU.SetSubVector(new double[] { n*Lsection, Wi/2.0}, new int[] { 1, -1 });
    pointsU.SetSubVector(new double[] { n*Lsection, We/2.0 }, new int[] { 2, -1 });

    particles.Add(new ParticleTriangle(noMotion, pointsU));
}


### outlet

In [ ]:
int OutletStreamWiseResolution = 6 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(nSections * Lsection, Lend, OutletStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid3D.Cartesian3DGrid(xNodes1, yNodes1, zNodes);

In [ ]:
double[] xNodes2 = xNodes1;
double[] yNodes2 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd2 = Grid3D.Cartesian3DGrid(xNodes2, yNodes2, zNodes);

In [ ]:
double[] xNodes3 = xNodes1;
double[] yNodes3 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid3D.Cartesian3DGrid(xNodes3, yNodes3, zNodes);

In [ ]:
var gridOutlet = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3});
//var gridOutletSealed = GridCommons.Seal(gridOulet, 4);

### System

In [ ]:
var gridSystem = GridCommons.MergeLogically(new GridCommons[] {gridInlet, gridSections, gridOutlet});
var grd = GridCommons.Seal(gridSystem, 4);

In [ ]:
grd.DefineEdgeTags(delegate (Vector X) {
    string ret = null;

    if ((X.y + We / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y - We / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();

    if ((X.x + Linlet).Abs() <= 1e-8)
        ret = IncompressibleBcType.Velocity_Inlet.ToString();
    if ((X.x - Lend).Abs() <= 1e-8)
        ret = IncompressibleBcType.Pressure_Outlet.ToString();

    if ((X.z + H / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.z - H / 2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();

    return ret;
});

Grid Edge Tags changed.


In [ ]:
grd.NumberOfCells

408

## Set up Controls 

In [ ]:
int k = 3;

In [ ]:
List<XNSERO_Control> Controls = new List<XNSERO_Control>();
Controls.Clear();

In [ ]:
XNSERO_Control C = new XNSERO_Control();

C.SetDGdegree(k);
C.SetGrid(grd);

Formula InletVelocity = new Formula("X => 0.25");   // roughly 1/4 of averBulkVelocity
C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", InletVelocity);

C.Particles = particles;
C.Option_LevelSetEvolution2 = LevelSetEvolution.None;

// C.AdaptiveMeshRefinement = true;
// C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { levelSet = 1, maxRefinementLevel = 1 });
// C.AMR_startUpSweeps = 1;

C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = viscosity;
C.PhysicalParameters.IncludeConvection = true;


C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
C.Option_LevelSetEvolution = LevelSetEvolution.None;


C.SessionName = "SCMC_SetupTestSystemRO_nSec2";

Controls.Add(C);

## Launch Jobs

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,SCMC_SetupTestSystemRO_nSec2_Stokes


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();

    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 4;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    oneJob.Activate(myBatch); 
}

Deployments so far (0): ;
Success: 0
job submit count: 0
unable to determine job status - unknown
Deploying job SCMC_SetupTestSystemRO_nSec2 ... 
Opening existing database 'D:\local\SharpCornerMicroChannel3D_RO'.
Opening existing database '\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\SharpCornerMicroChannel3D_RO'.
Set Database: { Session Count = 2; Grid Count = 2; Path = \\dc3\userspace\smuda\hpccluster\SharpCornerMicroChannel3D_RO }
Grid is not in database yet...
Grid successfully saved: 12be56aa-b729-40eb-9380-ddea8ef876fd
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\SharpCornerMicroChannel3D_RO-XNSERO_Solver2025Feb03_141322.479445
copied 46 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Postprocessing

In [ ]:
wmg.Sessions

In [ ]:
var sess = wmg.Sessions.Pick(0);
sess

SharpCornerMicroChannel	SCMC_SetupTestSystemRO_nSec3	01/29/2025 14:57:00	fa7ca4dd...

In [ ]:
sess.Export().WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\SharpCornerMicroChannel__SCMC_SetupTestSystemRO_nSec3__fa7ca4dd-cf56-4fb5-a2ed-f0497d270994
